# Week 4 Assignment
Subqueries, CTEs and Window Functions using Superstore Dataset

## Setup and Data Loading

In [1]:
import pandas as pd
import sqlite3
import os

# Create database connection
conn = sqlite3.connect("superstore.db")

print("Database created successfully.")

df = pd.read_csv(
    "Sample - Superstore.csv",
    encoding="latin1"      # use only if UTF-8 error occurs
)

df.to_sql(
    "superstore_raw",
    conn,
    if_exists="replace",
    index=False
)

print("superstore_raw table created.")


Database created successfully.
superstore_raw table created.


In [2]:
cursor = conn.cursor()
cursor.execute("""CREATE TABLE customers AS SELECT DISTINCT `Customer ID`,`Customer Name`,Segment FROM superstore_raw""")
cursor.execute("""CREATE TABLE orders AS SELECT DISTINCT `Order ID`,`Order Date`,`Ship Date`,`Ship Mode`,`Customer ID`,Sales,Quantity,Discount,Profit FROM superstore_raw""")
cursor.execute("""CREATE TABLE products AS SELECT DISTINCT `Product ID`,Category,`Sub-Category`,`Product Name` FROM superstore_raw""")
conn.commit()

## Query 1: Find all orders where sales are greater than the average sales

In [3]:
query1 = """
SELECT *
FROM orders
WHERE Sales >
(
    SELECT AVG(Sales)
    FROM orders
)
"""

pd.read_sql(query1, conn).head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,907.1520,6,0.20,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,1706.1840,9,0.20,85.3092


## Query 2: Find the highest sales order for each customer.

In [4]:
query = """
SELECT *
FROM orders o
WHERE Sales = (
    SELECT MAX(Sales)
    FROM orders
    WHERE `Customer ID` = o.`Customer ID`
)
"""
pd.read_sql(query, conn).head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.9400,3,0.00,219.5820
1,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.5775,5,0.45,-383.0310
2,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,1706.1840,9,0.20,85.3092
3,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,1044.6300,3,0.00,240.2649
4,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,3083.4300,7,0.50,-1665.0522


## Query 3: Calculate total sales for each customer. 

In [5]:
query3 = """
WITH customer_sales AS
(
    SELECT
        `Customer ID`,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY `Customer ID`
)

SELECT *
FROM customer_sales
ORDER BY Total_Sales DESC
"""

pd.read_sql(query3, conn).head()

,Customer ID,Total_Sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571


## Query 4: Find customers whose total sales are above average.

In [6]:
query4 = """
WITH customer_sales AS
(
    SELECT
        `Customer ID`,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY `Customer ID`
)

SELECT *
FROM customer_sales
WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
)
"""

pd.read_sql(query4, conn).head()

,Customer ID,Total_Sales
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846


## Query 5: Rank all customers based on total sales.

In [7]:
query5 = """
WITH customer_sales AS
(
    SELECT
        `Customer ID`,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY `Customer ID`
)

SELECT
    `Customer ID`,
    Total_Sales,
    RANK() OVER
    (
        ORDER BY Total_Sales DESC
    ) AS Customer_Rank
FROM customer_sales
"""

pd.read_sql(query5, conn).head()

,Customer ID,Total_Sales,Customer_Rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5


## Query 6: Assign row numbers to each order within a customer. 

In [8]:
query6 = """
SELECT
    `Customer ID`,
    `Order ID`,
    Sales,

    ROW_NUMBER() OVER
    (
        PARTITION BY `Customer ID`
        ORDER BY Sales DESC
    ) AS Order_Number

FROM orders
"""

pd.read_sql(query6, conn).head()

,Customer ID,Order ID,Sales,Order_Number
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2014-128055,673.568,2
2,AA-10315,CA-2016-103982,431.976,3
3,AA-10315,CA-2017-147039,362.940,4
4,AA-10315,CA-2014-128055,52.980,5


## Query 7: Display top 3 customers based on total sales.

In [9]:
query7 = """
WITH customer_sales AS
(
    SELECT
        `Customer ID`,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY `Customer ID`
),

ranked_customers AS
(
    SELECT
        `Customer ID`,
        Total_Sales,
        RANK() OVER
        (
            ORDER BY Total_Sales DESC
        ) AS Rank_No
    FROM customer_sales
)

SELECT *
FROM ranked_customers
WHERE Rank_No <= 3
"""

pd.read_sql(query7, conn)

,Customer ID,Total_Sales,Rank_No
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


## Final Combined Query
Write one final query that shows: 
• Customer Name  
• Total Sales  
• Rank 

In [10]:
final_query = """
WITH customer_sales AS
(
    SELECT
        `Customer ID`,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY `Customer ID`
),

ranked_customers AS
(
    SELECT
        `Customer ID`,
        Total_Sales,

        RANK() OVER
        (
            ORDER BY Total_Sales DESC
        ) AS Customer_Rank

    FROM customer_sales
)

SELECT
    c.`Customer Name`,
    r.Total_Sales,
    r.Customer_Rank

FROM ranked_customers r

JOIN customers c
ON r.`Customer ID` = c.`Customer ID`

ORDER BY Customer_Rank
"""

pd.read_sql(final_query, conn).head(10)

,Customer Name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
5,Ken Lonsdale,14175.229,6
6,Sanjit Chand,14142.334,7
7,Hunter Lopez,12873.298,8
8,Sanjit Engle,12209.438,9
9,Christopher Conant,12129.072,10
